# LLM-AL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import random
import logging
import time 
import yaml
import cohere
import seaborn as sns
import json
import glob
import re

# ignore warnings
import warnings
warnings.filterwarnings("ignore")


from openai import OpenAI
openrouter_api = "your_openrouter_api_key_here"
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=openrouter_api,
)

cohere_apiKey = 'your_cohere_api_key_here'
co = cohere.ClientV2(cohere_apiKey)

## Membrane Dataset

In [ ]:
data_grouped = pd.read_csv('Membrane_dataset.csv')
data_grouped

In [ ]:
ranges = 'Based on the dataset of experimental parameters for automated membrane synthesis via NIPS, I can identify the following parameter ranges:\n\n1. Index: Numerical identifiers for each experiment\n2. Auto: Binary parameter (Yes/No) - whether synthesis is automated or manual\n3. Heating: Binary parameter (Yes/No) - whether solution mixing involves heating\n4. Concentration: Range appears to be polymer concentrations in percentage (likely 10-20%)\n5. Heating block temperature: Temperature values in degrees (likely Celsius)\n6. Mixed: Binary parameter (Yes/No) - whether solution is robot-mixed or manually premixed\n7. Rest time: Number of days (0 to several days) solution rests before casting\n8. Coupon-to-Bath Wait Time: Time in minutes (likely 0-10 minutes) between casting and NIPS bath immersion\n9. Relative Humidity: Percentage values (likely 20-70% RH)\n10. Nitrogen (Side from drop): Binary parameter (Yes/No) - whether laminar nitrogen flow is used from side during drop casting\n11. Nitrogen (After blade): Binary parameter (Yes/No) - whether laminar nitrogen flow is used after blade casting\n\nWithout seeing the actual data values, these are the parameter types and likely ranges based on typical membrane synthesis conditions.'

### Report

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run=1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):
    random.seed(random_seed)

    # Store observations and results
    list_of_observations = []  
    list_of_modulus = []  
    trajectory_data = []
    # Copy dataset so original data remains intact
    data_copy = data_grouped.copy()  
    max_modulus = data_copy['Elastic Modulus_mean'].max()
    # Start by randomly selecting an initial experiment
    
    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Report with output'])
    list_of_modulus.append(data_copy.loc[initial_idx, 'Elastic Modulus_mean'])

    # Save initial selection in trajectory_data
    trajectory_data.append({
        "Iteration": 0,  # Iteration 0 for initial selection
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",  # No suggestion yet
        "Suggested Parameters": "N/A",  # No suggestion yet
        "Observed Elastic Modulus": data_copy.loc[initial_idx, 'Elastic Modulus_mean'],
        "Max Modulus in Dataset": max_modulus,
        "Stopping Reason": "Initial selection"
    })
    data_copy = data_copy.drop(index=initial_idx)
    

    def LLM_AL(observations_str, ranges, Model="anthropic/claude-3.7-sonnet", Temperature=0.0, sleep=0.5):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {"role": "system", "content": 
                    f"Here I have a set of structured experimental parameters for my automated membrane synthesis via non-solvent-induced phase separation (NIPS). "
                    f"My goal is to iteratively find the best experiment to maximize modulus. "
                    f"Based on the following previous experimental observations, recommend the next experiment that maximizes modulus. "
                    f"Suggest the next set of experimental parameters that is expected to maximize modulus given the parameter ranges: {ranges}. "
                    f"Do not suggest anything outside of these ranges or any parameters that are not mentioned."
                },
                {"role": "user", "content": f'\nPrior Observations: {observations_str}'}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    # Iteratively select the next best experiment
    for i in range(len(data_copy)):
        # Convert observations into a single string for query
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        # Prepare documents for reranking
        docs = list(data_copy['Report'])

        # Use Cohere's rerank model
        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)
        
        # Extract the best next experiment's index (position-based)
        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]  # Retrieve the experiment data
        best_modulus = best_experiment['Elastic Modulus_mean']

        # Save trajectory data
        trajectory_data.append({
            "Iteration": i+1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Formatted_Parameters'],
            "Suggested Parameters": suggestion,
            "Observed Elastic Modulus": best_modulus,
            "Max Modulus in Dataset": max_modulus,
            "Stopping Reason": "Max modulus reached" if best_modulus >= max_modulus else "Continuing"
        })


        # Store the newly selected experiment's results
        list_of_observations.append(best_experiment['Report with output'])
        list_of_modulus.append(best_modulus)

        # Remove the selected experiment from the dataset
        data_copy = data_copy.drop(data_copy.index[best_idx])

        # Early Stopping Condition: If no improvement, stop
        if best_modulus >= max_modulus:
            print(f"Stopping early at iteration {i+1} - Max modulus found.")
            break  


        # Print progress
        print(f"Iteration {i+1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Elastic Modulus: {best_modulus}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_membrane_modulus_trajectory_report_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    # Generate plot
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_modulus) + 1), list_of_modulus, marker='o', linestyle='-')
    plt.axhline(y=max_modulus, color='r', linestyle='--', label='Max Modulus')
    plt.xlabel("Iteration")
    plt.ylabel("Elastic Modulus")
    plt.title("Elastic Modulus Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df

In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)

### Parameters

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run = 1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):
    random.seed(random_seed)

    # Store observations and results
    list_of_observations = []  
    list_of_modulus = []  
    trajectory_data = []
    # Copy dataset so original data remains intact
    data_copy = data_grouped.copy()  
    max_modulus = data_copy['Elastic Modulus_mean'].max()
    # Start by randomly selecting an initial experiment
    
    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Formatted_Parameters with output'])
    list_of_modulus.append(data_copy.loc[initial_idx, 'Elastic Modulus_mean'])

    # Save initial selection in trajectory_data
    trajectory_data.append({
        "Iteration": 0,  # Iteration 0 for initial selection
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",  # No suggestion yet
        "Suggested Parameters": "N/A",  # No suggestion yet
        "Observed Elastic Modulus": data_copy.loc[initial_idx, 'Elastic Modulus_mean'],
        "Max Modulus in Dataset": max_modulus,
        "Stopping Reason": "Initial selection"
    })
    data_copy = data_copy.drop(index=initial_idx)
    

    def LLM_AL(observations_str, ranges, Model="anthropic/claude-3.7-sonnet", Temperature=0.0, sleep=0.5):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {"role": "system", "content": 
                    f"Here I have a set of structured experimental parameters for my automated membrane synthesis via non-solvent-induced phase separation (NIPS). "
                    f"My goal is to iteratively find the best experiment to maximize modulus. "
                    f"Based on the following previous experimental observations, recommend the next experiment that maximizes modulus. "
                    f"Suggest the next set of experimental parameters that is expected to maximize modulus given the parameter ranges: {ranges}. "
                    f"Do not suggest anything outside of these ranges or any parameters that are not mentioned."
                },
                {"role": "user", "content": f'\nPrior Observations: {observations_str}'}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    # Iteratively select the next best experiment
    for i in range(len(data_copy)):
        # Convert observations into a single string for query
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        # Prepare documents for reranking
        docs = list(data_copy['Formatted_Parameters'])

        # Use Cohere's rerank model
        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)
        
        # Extract the best next experiment's index (position-based)
        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]  # Retrieve the experiment data
        best_modulus = best_experiment['Elastic Modulus_mean']

        # Save trajectory data
        trajectory_data.append({
            "Iteration": i+1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Formatted_Parameters'],
            "Suggested Parameters": suggestion,
            "Observed Elastic Modulus": best_modulus,
            "Max Modulus in Dataset": max_modulus,
            "Stopping Reason": "Max modulus reached" if best_modulus >= max_modulus else "Continuing"
        })


        # Store the newly selected experiment's results
        list_of_observations.append(best_experiment['Formatted_Parameters with output'])
        list_of_modulus.append(best_modulus)

        # Remove the selected experiment from the dataset
        data_copy = data_copy.drop(data_copy.index[best_idx])

        # Early Stopping Condition: If no improvement, stop
        if best_modulus >= max_modulus:
            print(f"Stopping early at iteration {i+1} - Max modulus found.")
            break  


        # Print progress
        print(f"Iteration {i+1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Elastic Modulus: {best_modulus}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_membrane_modulus_trajectory_parameters_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    # Generate plot
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_modulus) + 1), list_of_modulus, marker='o', linestyle='-')
    plt.axhline(y=max_modulus, color='r', linestyle='--', label='Max Modulus')
    plt.xlabel("Iteration")
    plt.ylabel("Elastic Modulus")
    plt.title("Elastic Modulus Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df

In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)

## Matbench_steels

In [ ]:
data_grouped = pd.read_csv("matbench_steels.csv")
data_grouped

In [ ]:
# Elements of interest
elements = ['Fe', 'C', 'Mn', 'Si', 'Cr', 'Ni', 'Mo', 'V', 'Nb', 'Co', 'Al', 'Ti', 'N', 'W']

# Calculate min and max for each element
element_ranges = {
    elem: (data_grouped[elem].min(), data_grouped[elem].max()) for elem in elements
}

# Generate a sentence summarizing the ranges
ranges = "; ".join(
    [f"{elem} ranges from {min_val:.6f} to {max_val:.6f}" for elem, (min_val, max_val) in element_ranges.items()]
) + "."

ranges


### Report

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run = 1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):

    random.seed(random_seed)

    list_of_observations = []
    list_of_strength = []
    trajectory_data = []

    data_copy = data_grouped.copy()
    max_strength = data_copy['yield strength'].max()

    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Report with output'])
    list_of_strength.append(data_copy.loc[initial_idx, 'yield strength'])
    trajectory_data.append({
        "Iteration": 0,
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",
        "Suggested Parameters": "N/A",
        "Observed Yield Strength": data_copy.loc[initial_idx, 'yield strength'],
        "Max Yield Strength in Dataset": max_strength,
        "Stopping Reason": "Initial selection"
    })
    data_copy = data_copy.drop(index=initial_idx)

    def LLM_AL(observations_str, ranges, Model=model, Temperature=temperature, sleep=sleep):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {"role": "system", "content": 
                    f"You are helping to find steel compositions that maximize yield strength. "
                    f"Based on the following previous experimental reports, suggest the next composition to test. "
                    f"Only suggest steel compositions within these element ranges: {ranges}. "
                    f"Do not include any elements not listed or values outside these ranges."
                },
                {"role": "user", "content": f'\nPrior Observations: {observations_str}'}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    for i in range(len(data_copy)):
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        docs = list(data_copy['Report'])

        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)

        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]
        best_strength = best_experiment['yield strength']

        trajectory_data.append({
            "Iteration": i + 1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Report'],
            "Suggested Parameters": suggestion,
            "Observed Yield Strength": best_strength,
            "Max Yield Strength in Dataset": max_strength,
            "Stopping Reason": "Max yield strength reached" if best_strength >= max_strength else "Continuing"
        })

        list_of_observations.append(best_experiment['Report with output'])
        list_of_strength.append(best_strength)
        data_copy = data_copy.drop(data_copy.index[best_idx])

        if best_strength >= max_strength:
            print(f"Stopping early at iteration {i+1} - Max yield strength found.")
            break

        print(f"Iteration {i+1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Yield Strength: {best_strength}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_yield_strength_trajectory_report_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_strength) + 1), list_of_strength, marker='o', linestyle='-')
    plt.axhline(y=max_strength, color='r', linestyle='--', label='Max Yield Strength')

    
    plt.xlabel("Iteration")
    plt.ylabel("Yield Strength")
    plt.title("Yield Strength Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df


In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)

### Parameters

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run=1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):

    random.seed(random_seed)

    list_of_observations = []
    list_of_strength = []
    trajectory_data = []

    data_copy = data_grouped.copy()
    max_strength = data_copy['yield strength'].max()

    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Formatted_Parameters with output'])
    list_of_strength.append(data_copy.loc[initial_idx, 'yield strength'])
    trajectory_data.append({
        "Iteration": 0,
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",
        "Suggested Parameters": "N/A",
        "Observed Yield Strength": data_copy.loc[initial_idx, 'yield strength'],
        "Max Yield Strength in Dataset": max_strength,
        "Stopping Reason": "Initial selection"
    })
    data_copy = data_copy.drop(index=initial_idx)

    def LLM_AL(observations_str, ranges, Model=model, Temperature=temperature, sleep=sleep):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {"role": "system", "content": 
                    f"You are helping to find steel compositions that maximize yield strength. "
                    f"Based on the following previous experimental reports, suggest the next composition to test. "
                    f"Only suggest steel compositions within these element ranges: {ranges}. "
                    f"Do not include any elements not listed or values outside these ranges."
                },
                {"role": "user", "content": f'\nPrior Observations: {observations_str}'}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    for i in range(len(data_copy)):
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        docs = list(data_copy['Formatted_Parameters'])

        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)

        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]
        best_strength = best_experiment['yield strength']

        trajectory_data.append({
            "Iteration": i + 1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Formatted_Parameters'],
            "Suggested Parameters": suggestion,
            "Observed Yield Strength": best_strength,
            "Max Yield Strength in Dataset": max_strength,
            "Stopping Reason": "Max yield strength reached" if best_strength >= max_strength else "Continuing"
        })

        list_of_observations.append(best_experiment['Formatted_Parameters with output'])
        list_of_strength.append(best_strength)
        data_copy = data_copy.drop(data_copy.index[best_idx])

        if best_strength >= max_strength:
            print(f"Stopping early at iteration {i+1} - Max yield strength found.")
            break

        print(f"Iteration {i+1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Yield Strength: {best_strength}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_yield_strength_trajectory_parameters_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_strength) + 1), list_of_strength, marker='o', linestyle='-')
    plt.axhline(y=max_strength, color='r', linestyle='--', label='Max Yield Strength')

    
    plt.xlabel("Iteration")
    plt.ylabel("Yield Strength")
    plt.title("Yield Strength Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df


In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)

## P3HT Dataset

In [ ]:
data_grouped = pd.read_csv("P3HT_dataset.csv")
data_grouped

In [ ]:
# Identify relevant columns for composition (excluding conductivity and text-based columns)
composition_cols = [
    'P3HT content (%)', 'D1 content (%)', 'D2 content (%)',
    'D6 content (%)', 'D8 content (%)'
]

# Calculate min and max for each component
component_ranges = {
    col: (data_grouped[col].min(), data_grouped[col].max()) for col in composition_cols
}

# Generate a sentence summarizing the ranges
ranges = "; ".join(
    [f"{col.replace(' content (%)', '')} ranges from {min_val:.2f}% to {max_val:.2f}%" 
     for col, (min_val, max_val) in component_ranges.items()]
) + "."

ranges


### Report

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run=1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):
    
    random.seed(random_seed)

    list_of_observations = []
    list_of_conductivity = []
    trajectory_data = []

    data_copy = data_grouped.copy()
    max_cond = data_copy['Conductivity (measured) (S/cm)'].max()

    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Report with output'])
    list_of_conductivity.append(data_copy.loc[initial_idx, 'Conductivity (measured) (S/cm)'])

    trajectory_data.append({
        "Iteration": 0,
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",
        "Suggested Parameters": "N/A",
        "Observed Conductivity": data_copy.loc[initial_idx, 'Conductivity (measured) (S/cm)'],
        "Max Conductivity in Dataset": max_cond,
        "Stopping Reason": "Initial selection"
    })

    data_copy = data_copy.drop(index=initial_idx)

    def LLM_AL(observations_str, ranges, Model=model, Temperature=temperature, sleep=sleep):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are helping to design P3HT/CNT thin films with high electrical conductivity. "
                        "Based on the following previous experimental reports, suggest the next composition to test. "
                        f"Only suggest compositions within these component ranges: {ranges}. "
                        "Only use P3HT, D1, D2, D6, and D8."
                    )
                },
                {"role": "user", "content": f"Prior Observations: {observations_str}"}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    for i in range(len(data_copy)):
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        docs = list(data_copy['Report'])

        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)

        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]
        best_cond = best_experiment['Conductivity (measured) (S/cm)']

        trajectory_data.append({
            "Iteration": i + 1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Report'],
            "Suggested Parameters": suggestion,
            "Observed Conductivity": best_cond,
            "Max Conductivity in Dataset": max_cond,
            "Stopping Reason": "Max conductivity reached" if best_cond >= max_cond else "Continuing"
        })

        list_of_observations.append(best_experiment['Report with output'])
        list_of_conductivity.append(best_cond)
        data_copy = data_copy.drop(data_copy.index[best_idx])

        if best_cond >= max_cond:
            print(f"Stopping early at iteration {i + 1} - Max conductivity found.")
            break

        print(f"Iteration {i + 1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Conductivity: {best_cond}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_p3ht_conductivity_trajectory_report_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_conductivity) + 1), list_of_conductivity, marker='o', linestyle='-')
    plt.axhline(y=max_cond, color='r', linestyle='--', label='Max Conductivity')
    plt.xlabel("Iteration")
    plt.ylabel("Conductivity (S/cm)")
    plt.title("Conductivity Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df


In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)

### Parameters

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run = 1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):
    
    random.seed(random_seed)

    list_of_observations = []
    list_of_conductivity = []
    trajectory_data = []

    data_copy = data_grouped.copy()
    max_cond = data_copy['Conductivity (measured) (S/cm)'].max()

    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Formatted_Parameters with output'])
    list_of_conductivity.append(data_copy.loc[initial_idx, 'Conductivity (measured) (S/cm)'])

    trajectory_data.append({
        "Iteration": 0,
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",
        "Suggested Parameters": "N/A",
        "Observed Conductivity": data_copy.loc[initial_idx, 'Conductivity (measured) (S/cm)'],
        "Max Conductivity in Dataset": max_cond,
        "Stopping Reason": "Initial selection"
    })

    data_copy = data_copy.drop(index=initial_idx)

    def LLM_AL(observations_str, ranges, Model=model, Temperature=temperature, sleep=sleep):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are helping to design P3HT/CNT thin films with high electrical conductivity. "
                        "Based on the following previous experimental reports, suggest the next composition to test. "
                        f"Only suggest compositions within these component ranges: {ranges}. "
                        "Only use P3HT, D1, D2, D6, and D8."
                    )
                },
                {"role": "user", "content": f"Prior Observations: {observations_str}"}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    for i in range(len(data_copy)):
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        docs = list(data_copy['Formatted_Parameters'])

        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)

        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]
        best_cond = best_experiment['Conductivity (measured) (S/cm)']

        trajectory_data.append({
            "Iteration": i + 1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Formatted_Parameters'],
            "Suggested Parameters": suggestion,
            "Observed Conductivity": best_cond,
            "Max Conductivity in Dataset": max_cond,
            "Stopping Reason": "Max conductivity reached" if best_cond >= max_cond else "Continuing"
        })

        list_of_observations.append(best_experiment['Formatted_Parameters with output'])
        list_of_conductivity.append(best_cond)
        data_copy = data_copy.drop(data_copy.index[best_idx])

        if best_cond >= max_cond:
            print(f"Stopping early at iteration {i + 1} - Max conductivity found.")
            break

        print(f"Iteration {i + 1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Conductivity: {best_cond}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_p3ht_conductivity_trajectory_parameters_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_conductivity) + 1), list_of_conductivity, marker='o', linestyle='-')
    plt.axhline(y=max_cond, color='r', linestyle='--', label='Max Conductivity')
    plt.xlabel("Iteration")
    plt.ylabel("Conductivity (S/cm)")
    plt.title("Conductivity Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df


In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)

## Perovskite Dataset

In [ ]:
data_grouped = pd.read_csv("Perovskite_dataset.csv")
data_grouped

In [ ]:
# Identify relevant columns for perovskite composition
composition_cols = ['CsPbI', 'FAPbI', 'MAPbI']

# Calculate min and max for each component
component_ranges = {
    col: (data_grouped[col].min(), data_grouped[col].max()) for col in composition_cols
}

# Generate a sentence summarizing the ranges
ranges = "; ".join(
    [f"{col} ranges from {min_val:.2f} to {max_val:.2f}" 
     for col, (min_val, max_val) in component_ranges.items()]
) + "."

print(ranges)


### Report

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run=1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):
    random.seed(random_seed)

    list_of_observations = []
    list_of_instability = []
    trajectory_data = []

    data_copy = data_grouped.copy()
    min_index = data_copy['Instability index'].min()

    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Report with output'])
    list_of_instability.append(data_copy.loc[initial_idx, 'Instability index'])

    trajectory_data.append({
        "Iteration": 0,
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",
        "Suggested Parameters": "N/A",
        "Observed Instability Index": data_copy.loc[initial_idx, 'Instability index'],
        "Min Instability Index in Dataset": min_index,
        "Stopping Reason": "Initial selection"
    })

    data_copy = data_copy.drop(index=initial_idx)

    def LLM_AL(observations_str, ranges, Model=model, Temperature=temperature, sleep=sleep):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are helping to explore perovskite compositions with improved stability. "
                        "Based on the following previous experimental reports, suggest the next composition to test. "
                        f"Only suggest compositions within these component ranges: {ranges}. "
                        "Only use CsPbI, FAPbI, and MAPbI."
                    )
                },
                {"role": "user", "content": f"Prior Observations: {observations_str}"}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    for i in range(len(data_copy)):
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        docs = list(data_copy['Report'])

        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)

        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]
        best_index = best_experiment['Instability index']

        trajectory_data.append({
            "Iteration": i + 1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Report'],
            "Suggested Parameters": suggestion,
            "Observed Instability Index": best_index,
            "Min Instability Index in Dataset": min_index,
            "Stopping Reason": "Min instability index reached" if best_index <= min_index else "Continuing"
        })

        list_of_observations.append(best_experiment['Report with output'])
        list_of_instability.append(best_index)
        data_copy = data_copy.drop(data_copy.index[best_idx])

        if best_index <= min_index:
            print(f"Stopping early at iteration {i + 1} - Min instability index found.")
            break

        print(f"Iteration {i + 1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Instability Index: {best_index}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_perovskite_instability_trajectory_report_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_instability) + 1), list_of_instability, marker='o', linestyle='-')
    plt.axhline(y=min_index, color='r', linestyle='--', label='Min Instability Index')
    plt.xlabel("Iteration")
    plt.ylabel("Instability Index")
    plt.title("Instability Index Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df


In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)

### Parameters

In [ ]:
def run_experiment(data_grouped, ranges, random_seed=42, run=1, model="anthropic/claude-3.7-sonnet", temperature=0.0, sleep=0.5):

    random.seed(random_seed)

    list_of_observations = []
    list_of_instability = []
    trajectory_data = []

    data_copy = data_grouped.copy()
    min_index = data_copy['Instability index'].min()

    initial_idx = random.choice(data_copy.index)
    print(f"Initial observation: Experiment {initial_idx}")
    list_of_observations.append(data_copy.loc[initial_idx, 'Formatted_Parameters with output'])
    list_of_instability.append(data_copy.loc[initial_idx, 'Instability index'])

    trajectory_data.append({
        "Iteration": 0,
        "Experiment Index": initial_idx,
        "Experiment Parameters": "Initial Selection",
        "Suggested Parameters": "N/A",
        "Observed Instability Index": data_copy.loc[initial_idx, 'Instability index'],
        "Min Instability Index in Dataset": min_index,
        "Stopping Reason": "Initial selection"
    })

    data_copy = data_copy.drop(index=initial_idx)

    def LLM_AL(observations_str, ranges, Model=model, Temperature=temperature, sleep=sleep):
        response = client.chat.completions.create(
            model=Model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are helping to explore perovskite compositions with improved stability. "
                        "Based on the following previous experimental reports, suggest the next composition to test. "
                        f"Only suggest compositions within these component ranges: {ranges}. "
                        "Only use CsPbI, FAPbI, and MAPbI."
                    )
                },
                {"role": "user", "content": f"Prior Observations: {observations_str}"}
            ],
            temperature=Temperature
        )
        time.sleep(sleep)
        return response.choices[0].message.content

    for i in range(len(data_copy)):
        observations_str = ' '.join(list_of_observations)
        suggestion = LLM_AL(observations_str, ranges)
        query = f"Which of the following experiment matches best with the suggestion? \nSuggestion: {suggestion}"

        docs = list(data_copy['Formatted_Parameters'])

        results = co.rerank(model="rerank-v3.5", query=query, documents=docs, top_n=1)

        best_idx = results.results[0].index
        best_experiment = data_copy.iloc[best_idx]
        best_index = best_experiment['Instability index']

        trajectory_data.append({
            "Iteration": i + 1,
            "Experiment Index": best_experiment.name,
            "Experiment Parameters": best_experiment['Formatted_Parameters'],
            "Suggested Parameters": suggestion,
            "Observed Instability Index": best_index,
            "Min Instability Index in Dataset": min_index,
            "Stopping Reason": "Min instability index reached" if best_index <= min_index else "Continuing"
        })

        list_of_observations.append(best_experiment['Formatted_Parameters with output'])
        list_of_instability.append(best_index)
        data_copy = data_copy.drop(data_copy.index[best_idx])

        if best_index <= min_index:
            print(f"Stopping early at iteration {i + 1} - Min instability index found.")
            break

        print(f"Iteration {i + 1}: Added new observation (Experiment {best_experiment.name})")
        print(f"Suggestion: {suggestion}")
        print(f"Instability Index: {best_index}\n")

    trajectory_df = pd.DataFrame(trajectory_data)
    filename = f"llm_experiment_perovskite_instability_trajectory_parameters_seed{random_seed}_run{run}.csv"
    trajectory_df.to_csv(filename, index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(list_of_instability) + 1), list_of_instability, marker='o', linestyle='-')
    plt.axhline(y=min_index, color='r', linestyle='--', label='Min Instability Index')
    plt.xlabel("Iteration")
    plt.ylabel("Instability Index")
    plt.title("Instability Index Trajectory with Random Seed " + str(random_seed))
    plt.legend()
    plt.grid(True)
    plt.show()

    return trajectory_df


In [ ]:
for random_seed in [42, 41, 40, 39, 38]:
    for run in range(1, 6):
        run_experiment(data_grouped, ranges, random_seed=random_seed, run = run)